# Predicting Student Health Risk - Object-Oriented EDA

This notebook explores the **Playground Series S6E7: Predicting Student Health Risk** dataset.

The goal is to understand target balance, missingness, feature behavior, categorical patterns, and likely modeling risks before building a baseline model. The notebook is intentionally object-oriented: loading, schema inspection, chart rendering, drift checks, and takeaway generation live in reusable classes.


## tl;dr

Run the notebook top-to-bottom first. The final takeaways are generated from executed data, so they reflect the files available in the current environment.

Expected checks:

- Confirm whether the target is strongly imbalanced.
- Identify missingness patterns across numeric and categorical columns.
- Compare numeric feature behavior by health condition.
- Inspect categorical target mix and train/test drift.
- Produce modeling notes for the first baseline.


## Context & Methods

### Key Assumptions

- The notebook must work on Kaggle, where input files usually live under `/kaggle/input/...`.
- The same notebook should also run locally against `dataset/` in this project folder.
- The task is treated as multi-class classification with `health_condition` as the target.
- Visualizations avoid non-standard plotting dependencies and use inline HTML/SVG for portability.


In [ ]:
from __future__ import annotations
from dataclasses import dataclass
from pathlib import Path
from typing import Iterable
import html, math
import numpy as np
import pandas as pd

try:
    from IPython.display import HTML, display
except Exception:
    HTML = None
    def display(obj): print(str(obj)[:1200])

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 80)
pd.set_option("display.float_format", lambda value: f"{value:,.3f}")


## Data

The input resolver checks common Kaggle locations first and then falls back to local folders. This keeps the notebook reusable without path edits.


In [ ]:
@dataclass(frozen=True)
class EDAConfig:
    competition_slug: str = "playground-series-s6e7"
    local_dataset_dir: Path = Path("../dataset")
    target: str = "health_condition"
    id_column: str = "id"
    random_state: int = 42
    max_hist_rows: int = 120_000
    max_category_levels: int = 12
    svg_width: int = 860

class DataLoader:
    def __init__(self, config: EDAConfig): self.config = config
    def resolve_input_dir(self) -> Path:
        candidates = [
            Path("/kaggle/input/predicting-student-health-risk"),
            Path("/kaggle/input") / self.config.competition_slug,
            Path("/kaggle/input/playground-series-s6e7"),
            self.config.local_dataset_dir,
            Path("dataset"),
        ]
        for path in candidates:
            if (path / "train.csv").exists() and (path / "test.csv").exists(): return path
        raise FileNotFoundError("Could not find train.csv and test.csv. Checked: " + ", ".join(map(str, candidates)))
    def load(self):
        path = self.resolve_input_dir()
        return pd.read_csv(path / "train.csv"), pd.read_csv(path / "test.csv"), pd.read_csv(path / "sample_submission.csv"), path

class NotebookDisplay:
    @staticmethod
    def html(content: str):
        display(HTML(content)) if HTML else print(content[:1200])
    @staticmethod
    def card(title: str, body: str):
        style = "border:1px solid #dbe3ef;border-radius:8px;padding:16px 18px;margin:12px 0;background:#fff;box-shadow:0 1px 2px rgba(15,23,42,.05);"
        NotebookDisplay.html(f"<div style='{style}'><div style='font-size:17px;font-weight:700;color:#0f172a;margin-bottom:8px'>{html.escape(title)}</div><div style='font-size:14px;line-height:1.55;color:#334155'>{body}</div></div>")

config = EDAConfig()
train, test, sample_submission, input_dir = DataLoader(config).load()
NotebookDisplay.card("Input resolved", f"Loaded data from <code>{html.escape(str(input_dir))}</code>. Train rows: <b>{len(train):,}</b>, test rows: <b>{len(test):,}</b>, submission rows: <b>{len(sample_submission):,}</b>.")


In [ ]:
print("Train shape:", train.shape)
print("Test shape:", test.shape)
print("Sample submission shape:", sample_submission.shape)
display(train.head())
display(test.head())
display(sample_submission.head())


### Schema Overview

The schema check separates identifiers, target, numeric features, and categorical features. This is the first pass at understanding what a baseline model will need to handle.


In [ ]:
class SchemaInspector:
    def __init__(self, config: EDAConfig, train: pd.DataFrame, test: pd.DataFrame):
        self.config, self.train, self.test = config, train, test
    @property
    def feature_columns(self): return [c for c in self.train.columns if c not in {self.config.id_column, self.config.target}]
    @property
    def numeric_columns(self): return [c for c in self.feature_columns if pd.api.types.is_numeric_dtype(self.train[c])]
    @property
    def categorical_columns(self): return [c for c in self.feature_columns if c not in self.numeric_columns]
    def summary(self):
        rows=[]
        for c in self.train.columns:
            rows.append({
                "column": c,
                "role": "id" if c == self.config.id_column else "target" if c == self.config.target else "numeric feature" if pd.api.types.is_numeric_dtype(self.train[c]) else "categorical feature",
                "train_dtype": str(self.train[c].dtype),
                "test_dtype": str(self.test[c].dtype) if c in self.test.columns else "-",
                "train_missing_%": self.train[c].isna().mean() * 100,
                "test_missing_%": self.test[c].isna().mean() * 100 if c in self.test.columns else np.nan,
                "train_unique": self.train[c].nunique(dropna=True),
                "test_unique": self.test[c].nunique(dropna=True) if c in self.test.columns else np.nan,
            })
        return pd.DataFrame(rows)

schema = SchemaInspector(config, train, test)
schema_summary = schema.summary()
display(schema_summary)
print("Numeric features:", schema.numeric_columns)
print("Categorical features:", schema.categorical_columns)


## Results

### 1. Target Distribution

Target balance is the first modeling constraint. If one class dominates, plain accuracy can be misleading and validation should include per-class diagnostics.


In [ ]:
class SVGCharts:
    def __init__(self, width=860):
        self.width = width
        self.palette = {"at-risk":"#3B82F6", "fit":"#10B981", "unhealthy":"#EF4444", "Missing":"#F59E0B", "train_missing_%":"#3B82F6", "test_missing_%":"#64748B"}
    def wrap(self, title, subtitle, svg):
        return '<div style="font-family:-apple-system,BlinkMacSystemFont,Segoe UI,sans-serif;border:1px solid #dbe3ef;border-radius:8px;padding:16px;margin:14px 0;background:#fff">' + f"<div style='font-size:18px;font-weight:700;color:#0f172a'>{html.escape(title)}</div><div style='font-size:13px;color:#64748b;margin:4px 0 12px 0'>{html.escape(subtitle)}</div>{svg}</div>"
    def bar(self, data, title, subtitle, suffix=""):
        values = data.fillna(0).astype(float); maxv=max(values.abs().max(),1); left=210; row_h=34; top=18; chart_w=self.width-left-80; parts=[]
        for i,(label,value) in enumerate(values.items()):
            y=top+i*row_h; w=chart_w*abs(value)/maxv; color=self.palette.get(str(label),"#2563EB")
            parts += [f'<text x="0" y="{y+20}" font-size="12" fill="#334155">{html.escape(str(label))}</text>', f'<rect x="{left}" y="{y+6}" width="{w:.1f}" height="18" rx="3" fill="{color}" opacity=".88"></rect>', f'<text x="{left+w+8:.1f}" y="{y+20}" font-size="12" fill="#0f172a">{value:,.2f}{html.escape(suffix)}</text>']
        return self.wrap(title, subtitle, f'<svg viewBox="0 0 {self.width} {top*2+row_h*len(values)}" width="100%">{"".join(parts)}</svg>')
    def grouped_bar(self, frame, title, subtitle, suffix="%"):
        left=210; top=24; group_h=76; bar_h=15; chart_w=self.width-left-80; maxv=max(float(frame.max().max()),1); parts=[]
        for r,(label,row) in enumerate(frame.iterrows()):
            base=top+r*group_h; parts.append(f'<text x="0" y="{base+15}" font-size="12" font-weight="700" fill="#334155">{html.escape(str(label))}</text>')
            for j,col in enumerate(frame.columns):
                value=float(row[col]); y=base+24+j*(bar_h+4); w=chart_w*value/maxv; color=self.palette.get(str(col),"#2563EB")
                parts += [f'<rect x="{left}" y="{y}" width="{w:.1f}" height="{bar_h}" rx="3" fill="{color}" opacity=".88"></rect>', f'<text x="{left+w+7:.1f}" y="{y+12}" font-size="11" fill="#0f172a">{html.escape(str(col))}: {value:,.1f}{html.escape(suffix)}</text>']
        return self.wrap(title, subtitle, f'<svg viewBox="0 0 {self.width} {top*2+group_h*len(frame)}" width="100%">{"".join(parts)}</svg>')
    def histogram_grid(self, frame, columns, title, subtitle, bins=24):
        columns=list(columns); cw=260; ch=120; gx=28; gy=46; per=3; height=math.ceil(len(columns)/per)*(ch+gy)+18; parts=[]
        for i,c in enumerate(columns):
            values=frame[c].dropna().to_numpy()
            if len(values)==0: continue
            counts,edges=np.histogram(values,bins=bins); maxc=max(counts.max(),1); x0=(i%per)*(cw+gx); y0=(i//per)*(ch+gy)+20
            parts.append(f'<text x="{x0}" y="{y0-6}" font-size="12" font-weight="700" fill="#0f172a">{html.escape(c)}</text>')
            for b,count in enumerate(counts):
                bw=cw/bins-1; bh=(ch-24)*count/maxc; x=x0+b*(cw/bins); y=y0+ch-20-bh; parts.append(f'<rect x="{x:.1f}" y="{y:.1f}" width="{bw:.1f}" height="{bh:.1f}" fill="#3B82F6" opacity=".78"></rect>')
            parts += [f'<line x1="{x0}" y1="{y0+ch-20}" x2="{x0+cw}" y2="{y0+ch-20}" stroke="#CBD5E1"/>', f'<text x="{x0}" y="{y0+ch-4}" font-size="10" fill="#64748b">{edges[0]:,.1f}</text>', f'<text x="{x0+cw-48}" y="{y0+ch-4}" font-size="10" fill="#64748b">{edges[-1]:,.1f}</text>']
        return self.wrap(title, subtitle, f'<svg viewBox="0 0 {self.width} {height}" width="100%">{"".join(parts)}</svg>')
    def heatmap(self, matrix, title, subtitle, suffix=""):
        left=170; top=36; cell_h=32; cell_w=min(86,max(52,(self.width-left-40)/max(len(matrix.columns),1))); height=top+cell_h*len(matrix.index)+40; max_abs=max(float(np.nanmax(np.abs(matrix.to_numpy()))),1e-9); parts=[]
        for j,col in enumerate(matrix.columns):
            x=left+j*cell_w; parts.append(f'<text x="{x+cell_w/2}" y="22" text-anchor="middle" font-size="11" fill="#334155">{html.escape(str(col))}</text>')
        for i,(idx,row) in enumerate(matrix.iterrows()):
            y=top+i*cell_h; parts.append(f'<text x="0" y="{y+21}" font-size="11" fill="#334155">{html.escape(str(idx))}</text>')
            for j,value in enumerate(row):
                x=left+j*cell_w; alpha=min(abs(float(value))/max_abs,1); color="239,68,68" if value<0 else "37,99,235"
                parts += [f'<rect x="{x}" y="{y}" width="{cell_w-2}" height="{cell_h-2}" rx="3" fill="rgba({color},{0.12+alpha*.72:.2f})"></rect>', f'<text x="{x+cell_w/2}" y="{y+20}" text-anchor="middle" font-size="10" fill="#0f172a">{float(value):.2f}{html.escape(suffix)}</text>']
        return self.wrap(title, subtitle, f'<svg viewBox="0 0 {self.width} {height}" width="100%">{"".join(parts)}</svg>')

charts = SVGCharts(config.svg_width)


In [ ]:
target_counts = train[config.target].value_counts()
target_percent = train[config.target].value_counts(normalize=True).mul(100)
display(pd.DataFrame({"rows": target_counts, "share_%": target_percent.round(3)}))
NotebookDisplay.html(charts.bar(target_percent, "Target class distribution", "Share of rows by health condition in the training data.", "%"))


**Observation.** A dominant majority class would make a naive accuracy baseline look deceptively strong. Validation should include per-class recall or a confusion matrix once modeling starts.


### 2. Missing Values

Missingness matters twice: it can carry signal, and it can create train/test drift. The next views compare missing rates between train and test.


In [ ]:
class MissingnessAnalyzer:
    def __init__(self, config, train, test): self.config, self.train, self.test = config, train, test
    def summary(self):
        rows=[]
        for c in [x for x in self.train.columns if x != self.config.target]:
            rows.append({"column":c, "train_missing_%":self.train[c].isna().mean()*100, "test_missing_%":self.test[c].isna().mean()*100 if c in self.test.columns else np.nan, "delta_test_minus_train_pp":((self.test[c].isna().mean()-self.train[c].isna().mean())*100) if c in self.test.columns else np.nan})
        return pd.DataFrame(rows).sort_values("train_missing_%", ascending=False)
missing_summary = MissingnessAnalyzer(config, train, test).summary()
display(missing_summary)
NotebookDisplay.html(charts.grouped_bar(missing_summary.set_index("column")[["train_missing_%","test_missing_%"]].head(14), "Missing values by column", "Top columns by training missing rate, compared with test missing rate."))


**Observation.** Columns with visible missing rates should usually get explicit missing indicators in a modeling notebook. If train and test missing rates are similar, the pattern is less likely to be a split artifact.


### 3. Numeric Feature Distributions

The histograms below show the global shape of numeric features. Skew, hard bounds, and unusual spikes are useful hints for preprocessing and model choice.


In [ ]:
hist_sample = train.sample(min(len(train), config.max_hist_rows), random_state=config.random_state)
numeric_summary = train[schema.numeric_columns].describe(percentiles=[.01,.05,.25,.5,.75,.95,.99]).T
numeric_summary["missing_%"] = train[schema.numeric_columns].isna().mean().mul(100)
display(numeric_summary)
NotebookDisplay.html(charts.histogram_grid(hist_sample, schema.numeric_columns, "Numeric feature distributions", f"Histograms use up to {config.max_hist_rows:,} sampled training rows for speed."))


**Observation.** Tree-based models are a natural first baseline because these numeric columns have bounded ranges, missing values, and likely non-linear interactions.


### 4. Numeric Features by Target

Class-conditional averages are a compact way to find features that may separate the target classes. This is not causal evidence; it is a modeling diagnostic.


In [ ]:
class TargetComparison:
    def __init__(self, config, train, numeric_columns, categorical_columns): self.config, self.train, self.numeric_columns, self.categorical_columns = config, train, numeric_columns, categorical_columns
    def numeric_mean_lift(self):
        grouped = self.train.groupby(self.config.target, dropna=False)[self.numeric_columns].mean(); overall = self.train[self.numeric_columns].mean()
        return grouped.div(overall).sub(1).mul(100).T
    def category_target_mix(self, column):
        table = pd.crosstab(self.train[column].fillna("Missing"), self.train[self.config.target], normalize="index").mul(100)
        return table.loc[table.sum(axis=1).sort_values(ascending=False).index]
comparison = TargetComparison(config, train, schema.numeric_columns, schema.categorical_columns)
numeric_mean_lift = comparison.numeric_mean_lift()
display(numeric_mean_lift.round(2))
NotebookDisplay.html(charts.heatmap(numeric_mean_lift.round(2), "Numeric mean lift by target class", "Each cell is percent difference from the overall feature mean. Positive values are above the full-train average.", "%"))


**Observation.** Large positive or negative cells indicate features worth preserving as-is. Small cells do not mean the feature is useless because interactions can still be predictive.


### 5. Categorical Feature Mix

For each categorical feature, the chart shows the target composition within each category level. This helps identify categories with different risk profiles and spots levels where missing values behave like a real segment.


In [ ]:
for column in schema.categorical_columns:
    mix = comparison.category_target_mix(column).head(config.max_category_levels)
    display(mix.round(2))
    NotebookDisplay.html(charts.grouped_bar(mix, f"Target mix within {column}", "Rows sum to 100%. Missing values are shown as their own level when present."))


**Observation.** Categorical features should be encoded with missing values preserved as an explicit category. This keeps potentially meaningful missingness available to the model.


### 6. Train/Test Drift Scan

A quick drift scan compares numeric means and categorical distributions between train and test. Large shifts would make local validation less trustworthy.


In [ ]:
class DriftScanner:
    def __init__(self, config, train, test, numeric_columns, categorical_columns): self.config, self.train, self.test, self.numeric_columns, self.categorical_columns = config, train, test, numeric_columns, categorical_columns
    def numeric_drift(self):
        rows=[]
        for c in self.numeric_columns:
            train_mean=self.train[c].mean(); test_mean=self.test[c].mean(); pooled_std=pd.concat([self.train[c], self.test[c]], ignore_index=True).std()
            rows.append({"column":c, "train_mean":train_mean, "test_mean":test_mean, "standardized_mean_diff":(test_mean-train_mean)/pooled_std if pooled_std else 0})
        return pd.DataFrame(rows).sort_values("standardized_mean_diff", key=lambda s:s.abs(), ascending=False)
    def categorical_drift(self):
        rows=[]
        for c in self.categorical_columns:
            train_share=self.train[c].fillna("Missing").value_counts(normalize=True); test_share=self.test[c].fillna("Missing").value_counts(normalize=True); levels=train_share.index.union(test_share.index)
            tvd=0.5*(train_share.reindex(levels, fill_value=0)-test_share.reindex(levels, fill_value=0)).abs().sum(); rows.append({"column":c, "total_variation_distance":tvd})
        return pd.DataFrame(rows).sort_values("total_variation_distance", ascending=False)
scanner = DriftScanner(config, train, test, schema.numeric_columns, schema.categorical_columns)
numeric_drift = scanner.numeric_drift(); categorical_drift = scanner.categorical_drift()
display(numeric_drift); display(categorical_drift)
NotebookDisplay.html(charts.bar(numeric_drift.set_index("column")["standardized_mean_diff"].abs(), "Numeric train/test drift", "Absolute standardized mean difference. Smaller values suggest train and test are similarly distributed."))
NotebookDisplay.html(charts.bar(categorical_drift.set_index("column")["total_variation_distance"].mul(100), "Categorical train/test drift", "Total variation distance between train and test category shares.", "%"))


**Observation.** If drift is small, a standard stratified validation split should be a reasonable first validation setup. If drift is large in a feature, inspect whether missingness or category levels explain the shift.


### 7. Feature Correlation Map

Correlation is only a linear diagnostic, but it quickly identifies redundant numeric features and pairs that may interact with the target.


In [ ]:
correlation = train[schema.numeric_columns].corr(method="spearman")
display(correlation.round(3))
NotebookDisplay.html(charts.heatmap(correlation.round(2), "Spearman correlation between numeric features", "Rank correlation is robust enough for a quick monotonic relationship scan."))


### 8. Simple Feature Ideas

The next small table creates a few transparent derived features for inspection only. These are candidates for a later modeling notebook, not guaranteed improvements.


In [ ]:
class FeatureSketcher:
    def __init__(self, frame): self.frame = frame
    def build(self):
        features = pd.DataFrame(index=self.frame.index)
        features["steps_per_exercise_min"] = self.frame["step_count"] / self.frame["exercise_duration"].replace(0, np.nan)
        features["calories_per_step"] = self.frame["calorie_expenditure"] / self.frame["step_count"].replace(0, np.nan)
        features["sleep_water_product"] = self.frame["sleep_duration"] * self.frame["water_intake"]
        features["bmi_heart_rate_product"] = self.frame["bmi"] * self.frame["heart_rate"]
        return features.replace([np.inf, -np.inf], np.nan)
sketched_features = FeatureSketcher(train).build()
sketch_summary = sketched_features.describe(percentiles=[.05,.25,.5,.75,.95]).T
sketch_summary["missing_%"] = sketched_features.isna().mean().mul(100)
display(sketch_summary)
NotebookDisplay.html(charts.histogram_grid(sketched_features.sample(min(len(sketched_features), config.max_hist_rows), random_state=config.random_state), sketched_features.columns, "Candidate derived feature distributions", "These features are simple ratios/products that may capture activity efficiency or combined health signals."))


## Takeaways

The cell below writes a compact summary from the executed notebook state. It should update automatically if the dataset changes.


In [ ]:
class TakeawayWriter:
    def __init__(self, config, train, test, missing_summary, numeric_drift, categorical_drift): self.config, self.train, self.test, self.missing_summary, self.numeric_drift, self.categorical_drift = config, train, test, missing_summary, numeric_drift, categorical_drift
    def render(self):
        target_share = self.train[self.config.target].value_counts(normalize=True).mul(100)
        majority_class = target_share.idxmax(); majority_share = target_share.max(); top_missing = self.missing_summary.iloc[0]; top_num = self.numeric_drift.iloc[0]; top_cat = self.categorical_drift.iloc[0]
        items = [
            f"The training set has <b>{len(self.train):,}</b> rows and the test set has <b>{len(self.test):,}</b> rows.",
            f"The majority target class is <b>{html.escape(str(majority_class))}</b> at <b>{majority_share:.2f}%</b> of training rows.",
            f"The highest training missing rate is <b>{top_missing['train_missing_%']:.2f}%</b> in <b>{html.escape(str(top_missing['column']))}</b>.",
            f"The largest numeric train/test mean shift is in <b>{html.escape(str(top_num['column']))}</b> with absolute standardized mean difference <b>{abs(top_num['standardized_mean_diff']):.4f}</b>.",
            f"The largest categorical distribution shift is in <b>{html.escape(str(top_cat['column']))}</b> with total variation distance <b>{top_cat['total_variation_distance']:.4f}</b>.",
            "A practical first model should preserve missingness, encode categorical features, use stratified validation, and report per-class metrics.",
        ]
        return "<ul style='line-height:1.65;margin:0;padding-left:20px'>" + "".join(f"<li>{item}</li>" for item in items) + "</ul>"
NotebookDisplay.card("EDA takeaways", TakeawayWriter(config, train, test, missing_summary, numeric_drift, categorical_drift).render())


## Next Steps

- Confirm the official Kaggle metric before optimizing models.
- Build a small baseline with stratified validation.
- Preserve missing indicators and categorical missing levels.
- Compare a plain baseline against a class-weighted or threshold-aware version if the metric rewards minority-class performance.
